# Solar pipeline
Krótka analiza wsadowa danych solarnych w PySpark.

## 1. Cel projektu
Porównanie prostego pipeline baseline i optimized oraz wpływu warstw bronze i silver na czas obliczeń.

## 2. Dane i batch processing
Dane są historyczne, więc używamy przetwarzania wsadowego.

In [1]:
from pathlib import Path

from IPython.display import display

from spark_radiation_analysis.config import BRONZE_CSV, PLOTS_DIR, SILVER_PREPARED_DIR, TABLES_DIR, ensure_directories
from spark_radiation_analysis.experiments import compare_bronze_vs_silver, run_all_experiments
from spark_radiation_analysis.plotting import (
    plot_bronze_vs_silver_times,
    plot_experiment_times,
    plot_monthly_ghi,
    plot_relative_humidity_correlations,
)
from spark_radiation_analysis.processing import (
    build_ghi_research_outputs,
    build_relative_humidity_correlation_table,
    load_and_prepare_data,
    save_pandas_table,
    save_prepared_data,
)
from spark_radiation_analysis.spark_session import create_spark_session

ensure_directories()
PROJECT_ROOT = Path.cwd().resolve()
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'BRONZE_CSV: {BRONZE_CSV}')
print(f'SILVER_PREPARED_DIR: {SILVER_PREPARED_DIR}')
print(f'TABLES_DIR: {TABLES_DIR}')
print(f'PLOTS_DIR: {PLOTS_DIR}')


PROJECT_ROOT: /home/mrnobody/PycharmProjects/spark-meteo-analysis
BRONZE_CSV: /home/mrnobody/PycharmProjects/spark-meteo-analysis/data/bronze/solar-radiation/2017_2019.csv
SILVER_PREPARED_DIR: /home/mrnobody/PycharmProjects/spark-meteo-analysis/data/silver/solar_radiation_prepared
TABLES_DIR: /home/mrnobody/PycharmProjects/spark-meteo-analysis/results/tables
PLOTS_DIR: /home/mrnobody/PycharmProjects/spark-meteo-analysis/results/plots


## 3. Spark setup

In [2]:
spark = create_spark_session()
spark.sparkContext.setLogLevel('WARN')
assert BRONZE_CSV.exists(), f'Brak pliku: {BRONZE_CSV}'

26/06/12 21:48:53 WARN Utils: Your hostname, mrnobody-7A3 resolves to a loopback address: 127.0.1.1; using 192.168.1.35 instead (on interface enp6s0)
26/06/12 21:48:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 21:48:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 4. Data loading, preparation i zapis do silver

In [3]:
prepared_df = load_and_prepare_data(spark, BRONZE_CSV, use_schema=True)
silver_path = save_prepared_data(prepared_df, SILVER_PREPARED_DIR)
print('Liczba rekordów:', prepared_df.count())
print('Zapisano do:', silver_path)
prepared_df.select('timestamp', 'Month', 'GHI', 'DNI', 'DHI', 'Temperature', 'Relative_Humidity').show(5, truncate=False)

Liczba rekordów: 105120
Zapisano do: /home/mrnobody/PycharmProjects/spark-meteo-analysis/data/silver/solar_radiation_prepared
+-------------------+-----+---+---+---+-------------------+-----------------+
|timestamp          |Month|GHI|DNI|DHI|Temperature        |Relative_Humidity|
+-------------------+-----+---+---+---+-------------------+-----------------+
|2017-01-01 00:00:00|1    |0.0|0.0|0.0|-0.6000000000000001|86.29            |
|2017-01-01 00:15:00|1    |0.0|0.0|0.0|-0.6000000000000001|86.29            |
|2017-01-01 00:30:00|1    |0.0|0.0|0.0|-0.6000000000000001|86.29            |
|2017-01-01 00:45:00|1    |0.0|0.0|0.0|-0.6000000000000001|85.54            |
|2017-01-01 01:00:00|1    |0.0|0.0|0.0|-0.7000000000000001|86.17            |
+-------------------+-----+---+---+---+-------------------+-----------------+
only showing top 5 rows



## 5. Baseline i optimized experiment

In [4]:
baseline_monthly, optimized_monthly, metrics_df = run_all_experiments(spark, BRONZE_CSV)
display(metrics_df)
display(optimized_monthly)

,scenario,load_time_s,processing_time_s,total_pipeline_time_s,rows_processed,rows_per_second,partitions,speedup_vs_baseline
0,baseline,0.4424,0.3424,0.7848,105120,133944.36,1,1.0000
1,optimized,0.4282,0.1362,0.5644,105120,186256.70,1,1.3905


,Month,avg_ghi,avg_dni,avg_dhi,avg_temperature
0,1,164.492416,242.697472,69.093539,9.506096
1,2,244.102854,282.525007,108.782707,11.912179
2,3,353.033525,375.719173,137.205281,14.465557
3,4,455.094032,459.629025,150.872048,18.064856
4,5,457.671550,400.381072,176.656322,22.908462
5,6,517.137792,530.958605,143.660215,27.854983
6,7,562.362360,646.988427,116.367482,31.064940
7,8,534.367191,615.287586,124.024788,31.872148
8,9,465.653215,576.959872,117.525080,27.889945
9,10,369.905521,508.023804,110.717546,22.349006


## 6. Bronze vs silver

In [5]:
bronze_table, silver_table, bronze_silver_metrics_df = compare_bronze_vs_silver(spark, BRONZE_CSV, SILVER_PREPARED_DIR)
display(bronze_silver_metrics_df)
display(bronze_table)
display(silver_table)

,rows_processed,load_and_count_time_s,processing_time_s,total_time_s,rows_per_second,layer,speedup_vs_bronze
0,105120,0.1366,0.1853,0.3219,326591.31,bronze,1.0000
1,105120,0.0787,0.1186,0.1973,532709.27,silver,1.6315


,Month,avg_ghi,avg_dni,avg_dhi,avg_temperature
0,1,164.492416,242.697472,69.093539,9.506096
1,2,244.102854,282.525007,108.782707,11.912179
2,3,353.033525,375.719173,137.205281,14.465557
3,4,455.094032,459.629025,150.872048,18.064856
4,5,457.671550,400.381072,176.656322,22.908462
5,6,517.137792,530.958605,143.660215,27.854983
6,7,562.362360,646.988427,116.367482,31.064940
7,8,534.367191,615.287586,124.024788,31.872148
8,9,465.653215,576.959872,117.525080,27.889945
9,10,369.905521,508.023804,110.717546,22.349006


,Month,avg_ghi,avg_dni,avg_dhi,avg_temperature
0,1,164.492416,242.697472,69.093539,9.506096
1,2,244.102854,282.525007,108.782707,11.912179
2,3,353.033525,375.719173,137.205281,14.465557
3,4,455.094032,459.629025,150.872048,18.064856
4,5,457.671550,400.381072,176.656322,22.908462
5,6,517.137792,530.958605,143.660215,27.854983
6,7,562.362360,646.988427,116.367482,31.064940
7,8,534.367191,615.287586,124.024788,31.872148
8,9,465.653215,576.959872,117.525080,27.889945
9,10,369.905521,508.023804,110.717546,22.349006


## 7. Results tables

In [6]:
save_pandas_table(baseline_monthly, TABLES_DIR / 'baseline_monthly.csv')
save_pandas_table(optimized_monthly, TABLES_DIR / 'optimized_monthly.csv')
save_pandas_table(metrics_df, TABLES_DIR / 'experiment_metrics.csv')
save_pandas_table(bronze_silver_metrics_df, TABLES_DIR / 'bronze_vs_silver_metrics.csv')
print(TABLES_DIR)

/home/mrnobody/PycharmProjects/spark-meteo-analysis/results/tables


## 8. Results plots

In [7]:
plot_monthly_ghi(optimized_monthly, PLOTS_DIR / 'monthly_ghi.png')
plot_experiment_times(metrics_df, PLOTS_DIR / 'experiment_times.png')
plot_bronze_vs_silver_times(bronze_silver_metrics_df, PLOTS_DIR / 'bronze_vs_silver_times.png')
print(PLOTS_DIR)

/home/mrnobody/PycharmProjects/spark-meteo-analysis/results/plots


## 9. Small GHI research case
Pytanie: w których miesiącach i przy jakich warunkach pojawia się najwyższe GHI?

In [8]:
research_monthly_df, humidity_df, ghi_corr_df = build_ghi_research_outputs(prepared_df)
save_pandas_table(research_monthly_df, TABLES_DIR / 'research_monthly_ghi.csv')
save_pandas_table(humidity_df, TABLES_DIR / 'research_humidity_bands.csv')
save_pandas_table(ghi_corr_df, TABLES_DIR / 'research_ghi_correlations.csv')
display(research_monthly_df)
display(humidity_df)
display(ghi_corr_df)

,Month,avg_ghi,avg_temperature,avg_relative_humidity,avg_wind_speed
0,7,562.362360,31.064940,38.190330,3.707513
1,8,534.367191,31.872148,37.533281,3.448952
2,6,517.137792,27.854983,47.606202,3.012974
3,9,465.653215,27.889945,40.488242,3.270337
4,5,457.671550,22.908462,55.127542,2.688820
5,4,455.094032,18.064856,61.181458,2.790983
6,10,369.905521,22.349006,48.883764,2.837742
7,3,353.033525,14.465557,70.016243,3.482939
8,2,244.102854,11.912179,74.269802,3.418480
9,11,242.748031,17.457677,60.495782,3.024325


,humidity_band,avg_ghi,avg_temperature,rows
0,30-50,560.126798,27.185086,17098
1,50-70,357.976595,18.652630,15723
2,<30,733.200074,33.133432,5423
3,>=70,116.571081,13.467006,13963


,corr_ghi_temperature,corr_ghi_relative_humidity,corr_ghi_solar_zenith_angle,corr_ghi_wind_speed
0,0.576992,-0.725484,-0.816328,0.208753


## 10. Relative Humidity research case
Pytanie: które zmienne mają największy związek z Relative Humidity?

In [9]:
rh_corr_df = build_relative_humidity_correlation_table(prepared_df)
save_pandas_table(rh_corr_df, TABLES_DIR / 'relative_humidity_correlations.csv')
plot_relative_humidity_correlations(rh_corr_df, PLOTS_DIR / 'relative_humidity_correlations.png')
display(rh_corr_df)

26/06/12 21:48:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,feature,correlation_with_relative_humidity,abs_correlation
0,Temperature,-0.794409,0.794409
1,GHI,-0.725484,0.725484
2,DNI,-0.709469,0.709469
3,Solar_Zenith_Angle,0.666253,0.666253
4,DHI,-0.532825,0.532825
5,Pressure,0.185783,0.185783
6,Surface_Albedo,0.181516,0.181516
7,Wind_Speed,-0.138153,0.138153


## 11. Conclusions
- optimized jest szybszy od baseline,
- silver przyspiesza kolejne analizy względem bronze,
- najwyższe GHI występuje w miesiącach letnich,
- Relative Humidity najsilniej wiąże się z temperaturą, GHI i kątem zenitalnym.

In [10]:
spark.stop()